Before committing to a full re-embed (which takes hours), validate that the enrichment actually worked. Open this notebook and work through the steps.

# Step 1 — Load the Enriched Dataset

In [5]:
import pandas as pd

df = pd.read_csv('../data/processed/tracks_enriched.csv', encoding='cp1252')
print(df.shape)
print(df.columns.tolist())

(89740, 22)
['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'lastfm_tags']


# Step 2 — Check Tag Coverage

What to look for: You want at least 60–70% coverage. If it's much lower, either the enrichment script crashed early (load from checkpoint) or many tracks in the dataset are obscure enough that Last.fm has no tag data for them — that's normal and fine.

In [6]:
has_tags = df['lastfm_tags'].notna() & (df['lastfm_tags'] != '')
print(f"Tracks with tags: {has_tags.sum()} / {len(df)} ({has_tags.mean()*100:.1f}%)")

Tracks with tags: 3137 / 89740 (3.5%)


# Step 3 — Inspect Tag Vocabulary

What to look for: You should see genre tags (rock, pop, electronic), mood tags (chill, sad, happy), cultural tags (chinese, japanese, korean), and subculture tags (anime, vtuber, lofi). If cultural tags are sparse, the keyword boosting layer in 9.5 becomes the primary fallback.

In [7]:
from collections import Counter

all_tags = []
for tags_str in df['lastfm_tags'].dropna():
    all_tags.extend(tags_str.split())

tag_counts = Counter(all_tags)
print("Top 50 tags:")
for tag, count in tag_counts.most_common(50):
    print(f" {tag}: {count}")

Top 50 tags:
 rock: 231
 rockalternative: 180
 at: 172
 first: 167
 of: 161
 pop: 141
 indie: 133
 classic: 117
 rockpop: 105
 and: 103
 91.7: 91
 fm: 91
 the: 90
 metal: 76
 alternative: 74
 top: 73
 nu: 73
 hip: 70
 metalhard: 63
 punk: 61
 rockclassic: 61
 listen: 60
 a: 60
 rockhard: 55
 to: 52
 this: 51
 80snew: 49
 vocalists: 48
 hard: 48
 rap: 46
 metalrockalternative: 43
 rockindie: 41
 popindie: 39
 metalalternative: 38
 rockfemale: 37
 rockthe: 36
 song: 36
 with: 35
 rocklinkin: 34
 popfemale: 33
 my: 33
 new: 32
 for: 32
 dream: 31
 me: 31
 wsum: 30
 in: 30
 progressive: 30
 industrial: 29
 poppop: 29


# Step 4 — Spot-Check Specific Tracks

In [ ]:
def show_tags(track_name):
    rows = df[df['track_name'].str.lower() == track_name.lower()]
    for _, row in rows.iterrows():
        print(f"{row['track_name']} by {row['artists']}")
        print(f" Genre: {row['track_genre']}")
        print(f" Tags: {row['lastfm_tags']}")
        print()

show_tags("Bohemian Rhapsody")
show_tags("Clair de Lune")